# Dag 2: Environments + Command Jobs

**Nøglebegreber:** Environment (curated vs custom), command(), Input/Output

## 1. MLClient
Opret forbindelse til workspace

In [1]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
ml_client = init_ml_client()

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## 2. Curated Environment
En **curated environment** er en færdiglavet environment fra Microsoft.

**Opgave:** List alle curated environments med `ml_client.environments.list()` og find en der indeholder `sklearn`.

*Hint:* Curated environments har navne der starter med `AzureML-`.

In [7]:
for e in ml_client.environments.list():
    if 'AzureML-' in e.name and 'sklearn' in e.name:
        print(e)

creation_context:
  created_at: '2022-11-03T18:04:44.588235+00:00'
  created_by: Microsoft
  created_by_type: User
  last_modified_at: '2022-11-03T18:04:44.588235+00:00'
  last_modified_by: Microsoft
  last_modified_by_type: User
id: azureml:/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/environments/AzureML-sklearn-1.0-ubuntu20.04-py38-cpu
latest_version: '36'
name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu
tags: {}

creation_context:
  created_at: '2022-11-03T17:36:45.326893+00:00'
  created_by: Microsoft
  created_by_type: User
  last_modified_at: '2022-11-03T17:36:45.326893+00:00'
  last_modified_by: Microsoft
  last_modified_by_type: User
id: azureml:/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/environments/AzureML-sklearn-0.24-ubuntu18.04-py37-cpu

## 3. Custom Environment
En **custom environment** definerer du selv med enten:
- En conda specification (YAML)
- Et Docker image + conda
- Et Dockerfile

**Opgave:** Opret en custom environment med `Environment()` klassen.
- Brug `image` parameteren til at sætte et base Docker image (f.eks. `mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04`)
- Brug `conda_file` til at pege på en conda YAML fil
- Registrer den med `ml_client.environments.create_or_update()`

*Hint:* Du skal først oprette en `conda.yml` fil med dependencies som `scikit-learn`, `pandas`, og `mlflow`.

In [11]:
conda_spec = """
name: sklearn-1.5
channels:
- conda-forge
- anaconda
dependencies:
- python=3.10
- pip=25.3
- pandas~=1.5.3
- scipy~=1.10.0
- numpy~=1.22.0
- pip:
  - scikit-learn-intelex==2025.10.1
  - azureml-core==1.61.0.post1
  - azureml-defaults==1.61.0
  - azureml-mlflow==1.61.0.post1
  - azureml-telemetry==1.61.0
  - mlflow>=2.15,<3
  - scikit-learn~=1.5.0
  - joblib~=1.2.0
  # azureml-automl-common-tools packages
  - ipykernel~=6.0
  - tensorboard
  - psutil~=5.8.0
  - matplotlib~=3.5.0
  - tqdm~=4.66.3
  - py-cpuinfo==5.0.0
  - starlette>=0.49.1
"""

from pathlib import Path
Path("../conda_dependencies.yaml").write_text(conda_spec)

546

In [12]:
from azure.ai.ml.entities import Environment

custom_env = Environment(
    name="custom-environment",
    image="mcr.microsoft.com/azureml/openmpi5.0-ubuntu24.04:20260121.v1",
    conda_file="../conda_dependencies.yaml",
)

ml_client.environments.create_or_update(custom_env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi5.0-ubuntu24.04:20260121.v1', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'custom-environment', 'description': None, 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/environments/custom-environment/versions/3', 'Resource__source_path': '', 'base_path': '/Users/gade/Knowit/DP100/notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x16b54d100>, 'serialize': <msrest.serialization.Serializer object at 0x16941bf20>, 'version': '3', 'conda_file': {'channels': ['conda-forge', 'anaconda'], 'dependencies': ['python=3.10', 'pip=25.3', 'pandas~=1.5.3', 'scipy~=1.10.0', '

## 4. Træningsscript
Før vi kan køre et command job, skal vi have et `train.py` script.

**Opgave:** Lav `src/train.py` og udfyld scriptet så det:
1. Tager en `--input_data` argument med `argparse`
2. Læser CSV-filen med pandas
3. Træner en simpel model (f.eks. `LogisticRegression`) på churn-data
4. Printer accuracy

In [ ]:
%%writefile ../src/train.py
import argparse
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn

def make_dummies(df: pd.DataFrame, categorical_columns: list[str]):
    for col in categorical_columns:
        temp = df[col]
        dummies = pd.get_dummies(temp, prefix=col)
        df = pd.concat([df, dummies], axis=1)

    df.drop(columns=categorical_columns, inplace=True)

    return df

def get_data(path):
    df = pd.read_csv(path)

    # Count the rows and print the result
    row_count = (len(df))
    print('Analyzing {} rows of data'.format(row_count))
    
    return df

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--input_data", dest='input_data', type=str)
    parser.add_argument("--reg", dest='reg', type=float, default=0.01, help="Regularization rate (inverse used for C)")
    parser.add_argument("--model_dir", type=str, required=True, help="Directory to save model (AML output)")

    # parse args
    args = parser.parse_args()

    # return args
    return args

def main(args):
    df = get_data(args.input_data)

    keep_cols = ['Attrition', 'Age','Gender','Department','WorkLifeBalance','YearsSinceLastPromotion','JobInvolvement','YearsAtCompany','MonthlyIncome']
    df_reduced = df[keep_cols]
    categorical_cols = ['Gender', 'Department']
    df_reduced = make_dummies(df_reduced, categorical_columns=categorical_cols)
    X, y = df_reduced.drop(columns=['Attrition']).values, df_reduced['Attrition'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
    
    # set regularization hyperparameter
    C = 1.0 / float(args.reg)

    # train a logistic regression model
    print('Training a logistic regression model with regularization rate of', args.reg)
    model = LogisticRegression(C=C, solver="liblinear").fit(X_train, y_train)

    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("val_accuracy", acc)

    # Log hyperparams explicitly
    mlflow.log_param("reg", args.reg)
    mlflow.log_param("C", C)


    # AUC (if binary)
    if len(np.unique(y_test)) == 2:
        y_scores = model.predict_proba(X_test)[:, 1]
        auc = float(roc_auc_score(y_test, y_scores))
        print('AUC:', auc)
        mlflow.log_metric("val_auc", auc)
    else:
        auc = None

    # Save model with MLflow flavor metadata to the output folder
    out_dir = Path(args.model_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    mlflow.sklearn.save_model(model, str(out_dir / "model"))
    print(f"Saved MLflow model to {out_dir / 'model'}")

if __name__ == "__main__":
    args = parse_args()
    main(args)

In [9]:
%%writefile ../src/train.py
import argparse
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn

def make_dummies(df: pd.DataFrame, categorical_columns: list[str]):
    for col in categorical_columns:
        temp = df[col]
        dummies = pd.get_dummies(temp, prefix=col)
        df = pd.concat([df, dummies], axis=1)

    df.drop(columns=categorical_columns, inplace=True)

    return df

def get_data(path):
    df = pd.read_csv(path)

    # Count the rows and print the result
    row_count = (len(df))
    print('Analyzing {} rows of data'.format(row_count))

    return df

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--input_data", dest='input_data', type=str)
    parser.add_argument("--reg", dest='reg', type=float, default=0.01, help="Regularization rate (inverse used for C)")
    parser.add_argument("--model_dir", type=str, required=True, help="Directory to save model (AML output)")

    # parse args
    args = parser.parse_args()

    # return args
    return args

def main(args):
    df = get_data(args.input_data)

    keep_cols = ['Attrition', 'Age','Gender','Department','WorkLifeBalance','YearsSinceLastPromotion','JobInvolvement','YearsAtCompany','MonthlyIncome']
    df_reduced = df[keep_cols]
    categorical_cols = ['Gender', 'Department']
    df_reduced = make_dummies(df_reduced, categorical_columns=categorical_cols)
    X, y = df_reduced.drop(columns=['Attrition']).values, df_reduced['Attrition'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    # set regularization hyperparameter
    C = 1.0 / float(args.reg)

    # train a logistic regression model
    print('Training a logistic regression model with regularization rate of', args.reg)
    model = LogisticRegression(C=C, solver="liblinear").fit(X_train, y_train)

    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("val_accuracy", acc)

    # Log hyperparams explicitly
    mlflow.log_param("reg", args.reg)
    mlflow.log_param("C", C)


    # AUC (if binary)
    if len(np.unique(y_test)) == 2:
        y_scores = model.predict_proba(X_test)[:, 1]
        auc = float(roc_auc_score(y_test, y_scores))
        print('AUC:', auc)
        mlflow.log_metric("val_auc", auc)
    else:
        auc = None

    # Save model with MLflow flavor metadata to the output folder
    out_dir = Path(args.model_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    mlflow.sklearn.save_model(model, str(out_dir / "model"))
    print(f"Saved MLflow model to {out_dir / 'model'}")

if __name__ == "__main__":
    args = parse_args()
    main(args)

Overwriting ../src/train.py


## 5. Command Job
Et **command job** kører et script på remote compute.

**Opgave:** Brug `command()` funktionen til at oprette og submitte et job.
- `code`: stien til din `src/` mappe
- `command`: shell-kommandoen der kører scriptet, f.eks. `python train.py --input_data ${{inputs.data}}`
- `inputs`: dict med `Input` objekter — brug dit data asset fra Dag 1
- `environment`: brug enten curated eller din custom environment
- `compute`: `my-cluster`

*Hint:* `from azure.ai.ml import command, Input`

Submit med `ml_client.jobs.create_or_update(job)` og følg jobbet i Azure ML Studio.

In [10]:
from azure.ai.ml import command
from azure.ai.ml import Input, Output
from azure.ai.ml.constants import AssetTypes

job_inputs = {
    "data": Input(type=AssetTypes.URI_FILE, path="azureml:ibm-churn-file:1")
}

job_outputs = {
    "model": Output(type=AssetTypes.URI_FOLDER, path=None)
}

job_command = command(
    name="attrition-job-sklearn-new-env-2",
    description="Train a logistic regression in order to predict employee attrition",
    code="../src",
    environment="custom-environment:2",
    experiment_name="attrition-experiment",
    display_name="job-command-display-name",
    compute="my-cluster",
    command="python train.py --input_data ${{inputs.data}} --model_dir ${{outputs.model}}",
    inputs=job_inputs,
    outputs=job_outputs
)

ml_client.jobs.create_or_update(job_command)

Uploading src (0.0 MBs): 100%|██████████| 4184/4184 [00:00<00:00, 30024.05it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Experiment,Name,Type,Status,Details Page
attrition-experiment,attrition-job-sklearn-new-env-2,command,Starting,Link to Azure Machine Learning studio


## 6. (Bonus) Job med Output
**Opgave:** Udvid dit command job til også at have en `output`.
- Gem den trænede model til output-stien i dit script
- Definer `outputs` i `command()` med `Output(type=AssetTypes.URI_FOLDER)`
- Brug `${{outputs.model}}` i din command-streng